# 🎓 第3课：业界全景 —— 管理人怎么用因子？投资者怎么买因子？

**本节目标**：了解因子在真实世界怎么被使用，并亲手模拟「80% 收益来自 β」和「选 ETF 之难」。

> 对应原书：第1章 1.3 节

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
def _find_sample():
    for p in [Path('data/sample'), Path.cwd().parents[2] / 'data' / 'sample']:
        if p.exists():
            return p
    raise FileNotFoundError('找不到 data/sample，请从仓库根目录启动 notebook')
SAMPLE = _find_sample()
plt.rcParams['figure.dpi'] = 110
print('✅ 环境就绪，开始学习！')

✅ 环境就绪，开始学习！


## ① 管理人视角：5 件事 📋

| # | 关键词 | 白话 |
|---|---|---|
| 1 | 收益预测 vs 风险管理 | 因子既能选股（截面），也能算协方差控风险（时序） |
| 2 | 因子拥挤 | 用的人越多越不赚钱，出事时互相踩踏（2007年8月量化危机） |
| 3 | 因子择时 | 因子时灵时不灵，能躲开吗？（RA 按估值 vs AQR 按动量） |
| 4 | α vs β 收益 | 管理人赚的钱里 ~80% 来自 β（因子暴露），只有 20% 是真 α |
| 5 | 创新 | 另类数据（舆情/专利/新闻）+ 机器学习 = 新收益源 |

> 🛗 **电梯类比**：管理人收益像在电梯里做俯卧撑——电梯上行（β）贡献 80% 高度，俯卧撑（α）只占 20%。

## ② 案例1：「电梯类比」模拟 —— 谁才是顶尖管理人？🛗

模拟 100 个管理人：收益 = 80% 来自 β×市场 + 20% 来自 α 能力 + 运气。
看看「业绩最好的 10 个」是靠电梯（β）还是靠俯卧撑（α）？

In [2]:
rng = np.random.default_rng(5)
n = 100
beta = rng.uniform(0.8, 1.5, n)          # 电梯速度（因子暴露）
alpha = rng.normal(0, 0.001, n)          # 俯卧撑能力（真 α）
mkt = rng.normal(0.006, 0.04, 120)       # 市场收益
ret = 0.8 * beta[:, None] * mkt[None, :] + 0.2 * alpha[:, None] * 12 + rng.normal(0, 0.01, (n, 120))
ann = ret.mean(axis=1) * 12              # 年化收益

top = np.argsort(-ann)[:10]
pd.DataFrame({'年化收益%': (ann[top]*100).round(2),
             'β暴露(电梯)': beta[top].round(2),
             'α能力(俯卧撑)%': (alpha[top]*100).round(2)}).reset_index(drop=True)

,年化收益%,β暴露(电梯),α能力(俯卧撑)%
0,13.31,1.47,0.27
1,11.94,1.07,0.19
2,11.57,1.39,0.19
3,11.20,1.30,0.13
4,11.16,1.19,0.19
5,10.06,1.08,0.22
6,9.75,1.03,0.18
7,9.45,1.08,0.16
8,9.45,1.15,0.19
9,9.34,0.93,0.19


In [3]:
print('看懂了没？顶尖管理人几乎都是「电梯快」（β 高）的，而不是「俯卧撑强」（α 高）的！')
print('→ 评价基金先问：它的收益是选对了风格（β），还是真有选股能力（α）？')

看懂了没？顶尖管理人几乎都是「电梯快」（β 高）的，而不是「俯卧撑强」（α 高）的！
→ 评价基金先问：它的收益是选对了风格（β），还是真有选股能力（α）？


## ③ 投资者视角：从风格指数到 Smart Beta 🧺

- **1987** Russell 推出最早风格指数（价值/成长）；
- **1992** 先锋推出价值指数基金——主动投资的低成本替代；
- 今天 **Smart Beta ETF 规模超 6500 亿美元**（2018年估算）——普通投资者「一键买入因子」。

⚠️ 但选 ETF 极难！Choi et al. (2009) 实验：连金融专业学生都被历史收益迷惑，忽视费率。
下面复刻这个实验。

## ④ 案例2：复刻「选 ETF 实验」 🎯

5 个 ETF 追踪**同一个价值因子**，毛收益完全相同，只有**费率**不同（0.05%~0.50%）。
加上随机扰动模拟「历史表现差异」——你会选哪个？

In [4]:
rng = np.random.default_rng(1)
fees = [0.05, 0.10, 0.20, 0.35, 0.50]           # 年费率 %
factor_ret = rng.normal(0.006, 0.03, size=60)    # 价值因子月收益
noise = rng.normal(0, 0.002, size=(60, 5))       # 干扰：历史表现随机波动

# 「迷惑版」：毛收益 + 随机扰动（像真实市场一样让人看走眼）
gross = pd.DataFrame({f'ETF_{i+1}(费率{fee}%)': factor_ret - fee/100/12 + noise[:, i]
                      for i, fee in enumerate(fees)})
print('近5年累计收益（你会选哪个？）：')
print(((1+gross).prod()-1).round(3).sort_values(ascending=False))

近5年累计收益（你会选哪个？）：
ETF_5(费率0.5%)     0.273
ETF_1(费率0.05%)    0.270
ETF_2(费率0.1%)     0.269
ETF_3(费率0.2%)     0.234
ETF_4(费率0.35%)    0.220
dtype: float64


In [5]:
# 揭晓：扣掉费率后的真实收益——唯一标准是费率！
clean = pd.DataFrame({f'ETF_{i+1}(费率{fee}%)': factor_ret - fee/100/12 for i, fee in enumerate(fees)})
net = (1+clean).prod()-1
print('扣费后真实累计收益：')
print(net.round(3).sort_values(ascending=False))
best = net.idxmax()
print(f'\n🏆 赢家是 {best} —— 费率最低的才是对的！')

扣费后真实累计收益：
ETF_1(费率0.05%)    0.283
ETF_2(费率0.1%)     0.279
ETF_3(费率0.2%)     0.273
ETF_4(费率0.35%)    0.264
ETF_5(费率0.5%)     0.254
dtype: float64

🏆 赢家是 ETF_1(费率0.05%) —— 费率最低的才是对的！


## ✋ 试一试

1. 案例1中把 `0.8` 和 `0.2` 换成 `0.2` 和 `0.8` 重跑——顶尖管理人还看得到「电梯」吗？
2. 案例2把 `fees` 改成 [0.5, 0.5, 0.5, 0.5, 0.5] 重跑——结果变成什么？说明什么？

## ✅ 小结

- 管理人：**80% 收益来自 β**（选对风格），因子拥挤是最大风险；
- 投资者：Smart Beta 让普通人也能买因子，但**同类 ETF 之间只看费率**；
- 第1章完结！你已经有了完整的「因子世界观」。

> 下一章（第2章）：方法论——排序法、回归、Newey-West……把「感觉」变成「检验」。